# **Train CORE trên I3D & MediaPipe features (fairseq How2Sign / RWTH-PHOENIX)**

Notebook này train **CORE** (Transformer encoder → XE → SCST) trên **cả 2 bộ đặc trưng** bạn có
(`i3d` và `mediapipe`) trong **1 lần chạy**, rồi in bảng so sánh BLEU-4 i3d vs mediapipe.

**Vì sao chỉ CORE (transformer)?** Encoder transformer chỉ là `Linear(feat_dim → d_model)` nên nhận
MỌI chiều đặc trưng (I3D ~1024-d, MediaPipe khác). Các encoder đồ thị (gcn / graph_transformer) bị
khoá `assert pose_dim==183` (giả định layout khớp xương pose 183-d) nên KHÔNG chạy với feature này.

Notebook **tái dùng đúng code lõi của repo** (`SLTTransformer`, `train_xe`, `train_scst`, `evaluate`) -
chỉ thay data loader bằng bộ đọc **fairseq TSV manifest + file feature rời theo id**.

### Chuẩn bị (Kaggle)
| Mục | Giá trị |
|---|---|
| Accelerator | **GPU T4** |
| Internet | **ON** (để `git clone` repo + `pip install`) |
| Add Input | dataset chứa `i3d_features…`, `mediapipe_features…`, `tsv_files…` (How2Sign và/hoặc PHOENIX) |

### Cách dùng
1. Chỉnh **Cell CẤU HÌNH** (chọn `BASE`, `MODALITIES`, `SUBSET`, số epoch…).
2. Chạy tuần tự. **Cell INSPECT** sẽ tự dò TSV + feature, in `ID_COL / TEXT_COL / feat_dim` — **kiểm tra
   các giá trị này hợp lý** trước khi để nó train (nếu sai → gửi mình header + 1 dòng TSV để chỉnh loader).

In [ ]:
# ============================ CẤU HÌNH ============================
BASE        = "how2sign"            # lọc khi Add CẢ 2 dataset: "how2sign" | "phoenix" (alias "rwth")
MODALITIES  = ["i3d", "mediapipe"]  # train lần lượt cả 2 để so sánh; để 1 phần tử nếu chỉ muốn 1
SUBSET      = 0.10                  # % train split -> chọn 0.05 / 0.10 / 0.25 (dev/val + test LUÔN full).
                                    #   =1.0 = FULL How2Sign (~30k câu, RẤT lâu: SCST ~55 phút/epoch).
XE_EPOCHS   = 40                    # cross-entropy; có early-stop (patience=10) nên tự dừng nếu phẳng sớm.
RL_EPOCHS   = 10                    # SCST (đắt: ~15 phút/epoch ở 25%); 10 là đủ, cũng có early-stop.
RUN_SCST    = True                 # False = chỉ XE (nhanh hơn, chỉ đo BLEU của XE, bỏ RL)
BATCH       = 16
VOCAB       = 3000                 # subword BPE (SentencePiece) build từ text của train split
MAX_FRAMES  = 300                  # truncate chuỗi feature quá dài (uniform sample)

# Ước tính thời gian (CẢ 2 modality, T4, 40 XE + 10 SCST): 5%≈2.4h · 10%≈3.4h · 25%≈6.6h -> đều gọn 1 session.
# Chỉ điền khi auto-discovery ở Cell INSPECT báo SAI (để "" / {} = tự dò trong /kaggle/input):
TSV_TRAIN_OVERRIDE = ""; TSV_VAL_OVERRIDE = ""; TSV_TEST_OVERRIDE = ""
FEATDIR_OVERRIDE   = {}            # vd {"mediapipe": "/kaggle/input/…/mediapipe_features_how2sign/mediapipe_features"}
print("BASE =", BASE, "| MODALITIES =", MODALITIES, "| SUBSET =", SUBSET,
      "| XE_EPOCHS =", XE_EPOCHS, "| RL_EPOCHS =", RL_EPOCHS if RUN_SCST else "(bỏ)")

In [ ]:
# ---------- Cài deps + lấy code lõi từ GitHub ----------
import os, sys, subprocess
def _pip(*a): subprocess.run([sys.executable, "-m", "pip", *a], check=True)
_pip("install", "-q", "sentencepiece", "sacrebleu")

os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", "Sign-Language-REL_code"])
subprocess.run(["git", "clone", "https://github.com/linhxm/Sign-Language-REL_code.git"], check=True)
os.chdir("/kaggle/working/Sign-Language-REL_code")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

In [ ]:
# ---------- INSPECT: tự dò TSV + feature, phát hiện cột & chiều đặc trưng ----------
import os, glob
import numpy as np, pandas as pd, torch

BASE_KEYS = {"how2sign": ["how2sign"], "phoenix": ["phoenix", "rwth"], "rwth": ["rwth", "phoenix"]}
_bkeys = BASE_KEYS.get(BASE.lower(), [BASE.lower()])
def _match_base(p): return any(k in p.lower() for k in _bkeys)

SPLIT_ALTS = {"train": ["train"], "val": ["val", "valid", "dev"], "test": ["test"]}

def _load_feat(path):
    if path.lower().endswith(".npz"):
        z = np.load(path); arr = z[list(z.keys())[0]]
    elif path.lower().endswith(".pt"):
        arr = torch.load(path, map_location="cpu")
        arr = arr.numpy() if hasattr(arr, "numpy") else np.asarray(arr)
    else:
        arr = np.load(path)
    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim == 1:   arr = arr[None, :]
    elif arr.ndim > 2:  arr = arr.reshape(arr.shape[0], -1)   # vd MediaPipe [T,V,C] -> [T,V*C]
    return arr

def _glob_files(root, exts=(".npy", ".npz", ".pt"), limit=None):
    out = []
    for f in glob.glob(os.path.join(root, "**", "*"), recursive=True):
        if f.lower().endswith(exts):
            out.append(f)
            if limit and len(out) >= limit: break
    return out

def find_tsv(split, override):
    if override: return override
    hits = []
    for nm in SPLIT_ALTS[split]:
        hits += glob.glob("/kaggle/input/**/*%s*" % nm, recursive=True)
    hits = [h for h in hits if os.path.isfile(h) and "cvpr23" in os.path.basename(h).lower()]
    based = [h for h in hits if _match_base(h)]
    hits = based or hits
    assert hits, "Không thấy TSV manifest (tên chứa 'cvpr23') cho split=%s. Đã Add dataset TSV chưa?" % split
    return sorted(hits, key=len)[0]

def find_split_dir(modality, split, override_root=None):
    roots = []
    if override_root:
        for nm in SPLIT_ALTS[split]:
            d = os.path.join(override_root, nm)
            if os.path.isdir(d): roots.append(d)
    if not roots:
        # Khớp theo pattern '<modality>_features' (i3d_features..., mediapipe_features...) TRƯỚC.
        # Tránh dính tên DATASET chứa cả 'i3d' lẫn 'mediapipe' (vd how2sign-i3d-features-mediapipe-
        # features): nếu chỉ 'modality in path' thì i3d & mediapipe cùng trỏ 1 thư mục -> train TRÙNG.
        want = "%s_features" % modality.lower()
        for strict in (True, False):
            for nm in SPLIT_ALTS[split]:
                for d in glob.glob("/kaggle/input/**/%s" % nm, recursive=True):
                    if not (os.path.isdir(d) and _match_base(d)): continue
                    low = d.lower()
                    if (want in low) if strict else (modality.lower() in low):
                        roots.append(d)
            if roots: break   # match chặt ('_features') ra rồi thì dùng luôn, không rơi xuống lỏng
    roots = [d for d in roots if _glob_files(d, limit=1)]
    assert roots, ("Không thấy thư mục feature %s/%s có file .npy. Kiểm tra BASE / đã Add dataset? "
                   "Nếu thư mục không theo '<modality>_features', đặt FEATDIR_OVERRIDE[%r]."
                   % (modality, split, modality))
    return sorted(roots, key=len)[0]

TSV = {s: find_tsv(s, ov) for s, ov in
       [("train", TSV_TRAIN_OVERRIDE), ("val", TSV_VAL_OVERRIDE), ("test", TSV_TEST_OVERRIDE)]}
print("TSV manifests:")
for s, p in TSV.items(): print("  %-5s -> %s" % (s, p))

SPLIT_DIR = {mod: {s: find_split_dir(mod, s, FEATDIR_OVERRIDE.get(mod)) for s in ("train", "val", "test")}
             for mod in MODALITIES}
print("\nThư mục feature:")
for mod in MODALITIES:
    for s in ("train", "val", "test"): print("  %-10s %-5s -> %s" % (mod, s, SPLIT_DIR[mod][s]))

# GUARD: 2 modality KHÔNG được trỏ cùng 1 thư mục (chính là bug 'train i3d 2 lần' khi tên dataset
# chứa cả 'i3d' lẫn 'mediapipe'). Nếu trùng -> dừng NGAY, không train trùng một cách âm thầm.
_seen = {}
for mod in MODALITIES:
    key = os.path.normpath(SPLIT_DIR[mod]["train"])
    if key in _seen:
        raise AssertionError(
            "Modality %r và %r CÙNG trỏ về %s -> sẽ train TRÙNG (không phải so sánh thật). "
            "Thư mục mediapipe có thể tên khác / không tồn tại dạng .npy; đặt FEATDIR_OVERRIDE, vd "
            "{'mediapipe': '/kaggle/input/.../mediapipe_features_how2sign/mediapipe_features'}."
            % (_seen[key], mod, key))
    _seen[key] = mod

# --- phát hiện cột id + cột text ---
df0 = pd.read_csv(TSV["train"], sep="\t", quoting=3, keep_default_na=False, dtype=str)
print("\nCột TSV:", list(df0.columns))
print("Dòng đầu:", {k: (str(v)[:60] + "…" if len(str(v)) > 60 else v) for k, v in df0.iloc[0].to_dict().items()})

ref_stems = set(os.path.splitext(os.path.basename(f))[0]
                for f in _glob_files(SPLIT_DIR[MODALITIES[0]]["train"], limit=8000))
best = (None, -1)
for c in df0.columns:
    h = int(df0[c].astype(str).isin(ref_stems).sum())
    if h > best[1]: best = (c, h)
ID_COL = best[0]
assert best[1] > 0, ("Không map được cột nào với tên file feature.\n  Cột: %s\n  Ví dụ id file: %s\n"
                     "  -> gửi mình header + 1 dòng TSV để chỉnh loader." % (list(df0.columns), list(ref_stems)[:3]))
print("\n=> ID_COL   = %r  (khớp %d id file)" % (ID_COL, best[1]))

KNOWN = ["translation", "text", "tgt_text", "sentence", "target", "tgt", "caption", "gloss_free_text"]
cands = [c for c in df0.columns if c != ID_COL]
named = [c for c in cands if c.lower() in KNOWN]
TEXT_COL = named[0] if named else max(cands, key=lambda c: df0[c].astype(str).str.len().mean())
print("=> TEXT_COL = %r  (vd: %r)" % (TEXT_COL, str(df0[TEXT_COL].iloc[0])[:80]))

FEAT_DIM = {}
for mod in MODALITIES:
    s = _glob_files(SPLIT_DIR[mod]["train"], limit=1)[0]
    a = _load_feat(s); FEAT_DIM[mod] = int(a.shape[-1])
    print("=> [%s] feat_dim = %d  (sample shape %s, %s)" % (mod, a.shape[-1], tuple(a.shape), os.path.basename(s)))

print("\n✅ Inspect xong — KIỂM TRA ID_COL / TEXT_COL / feat_dim ở trên hợp lý rồi hãy chạy tiếp.")

In [ ]:
# ---------- Dataset đọc feature theo id + tokenizer từ TSV ----------
import warnings
from functools import partial
from torch.utils.data import Dataset, DataLoader
from data.dataset import collate_fn
from data.tokenizer import Tokenizer

def _build_index(split_dir):
    idx = {}
    for f in glob.glob(os.path.join(split_dir, "**", "*"), recursive=True):
        if f.lower().endswith((".npy", ".npz", ".pt")):
            idx[os.path.splitext(os.path.basename(f))[0]] = f
    return idx

class FairseqFeatureDataset(Dataset):
    """Đọc manifest fairseq TSV (id + text) + file feature rời theo id (I3D/MediaPipe).
    Trả về đúng batch-item mà collate_fn của repo cần: pose[T,D] / text_ids / text_raw / name."""
    def __init__(self, tsv_path, split_dir, tokenizer, id_col, text_col, feat_dim,
                 max_frames=300, max_text_len=60, subset_ratio=1.0, seed=42):
        df = pd.read_csv(tsv_path, sep="\t", quoting=3, keep_default_na=False, dtype=str)
        df = df[df[text_col].astype(str).str.strip() != ""].reset_index(drop=True)
        if subset_ratio < 1.0:
            df = df.sample(frac=subset_ratio, random_state=seed).reset_index(drop=True)
        self.index = _build_index(split_dir)
        keep = df[id_col].astype(str).isin(self.index)
        n_drop = int((~keep).sum())
        if n_drop:
            warnings.warn("[%s] %d/%d dòng KHÔNG có feature file -> bỏ. Nếu lớn: sai id_col hoặc sai "
                          "thư mục feature." % (os.path.basename(split_dir), n_drop, len(df)))
        self.df = df[keep].reset_index(drop=True)
        assert len(self.df) > 0, "0 mẫu khớp feature ở %s — kiểm tra id_col/split_dir." % split_dir
        self.tok, self.id_col, self.text_col = tokenizer, id_col, text_col
        self.feat_dim, self.max_frames, self.max_text_len = feat_dim, max_frames, max_text_len

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        name, text = str(row[self.id_col]), str(row[self.text_col])
        arr = _load_feat(self.index[name])
        if len(arr) > self.max_frames:
            sel = np.linspace(0, len(arr) - 1, self.max_frames).astype(int)
            arr = arr[sel]
        ids = self.tok.encode(text, add_special=True)
        if len(ids) > self.max_text_len:
            ids = ids[: self.max_text_len - 1] + [self.tok.eos_id]
        return {"pose": torch.from_numpy(np.ascontiguousarray(arr)),
                "text_ids": torch.tensor(ids, dtype=torch.long),
                "text_raw": text, "name": name}

def build_tokenizer_from_tsv(tsv_path, text_col, work_dir, vocab_size=3000):
    df = pd.read_csv(tsv_path, sep="\t", quoting=3, keep_default_na=False, dtype=str)
    txt = os.path.join(work_dir, "feat_train_text.txt")
    with open(txt, "w", encoding="utf-8") as f:
        for t in df[text_col].astype(str):
            t = t.strip()
            if t: f.write(t + "\n")
    prefix = os.path.join(work_dir, "spm_feat")
    Tokenizer.train(txt, prefix, vocab_size=vocab_size)
    return Tokenizer(prefix + ".model")

def make_loaders_feat(mod, tokenizer, subset):
    coll = partial(collate_fn, pad_id=tokenizer.pad_id)
    mk_ds = lambda split, sub: FairseqFeatureDataset(
        TSV[split], SPLIT_DIR[mod][split], tokenizer, ID_COL, TEXT_COL, FEAT_DIM[mod],
        MAX_FRAMES, 60, sub, 42)
    mk_ld = lambda ds, sh: DataLoader(ds, batch_size=BATCH, shuffle=sh, collate_fn=coll,
                                      num_workers=2, pin_memory=True)
    return (mk_ld(mk_ds("train", subset), True),
            mk_ld(mk_ds("val", 1.0), False),
            mk_ld(mk_ds("test", 1.0), False))
print("Dataset/loader sẵn sàng.")

In [ ]:
# ---------- TRAIN: XE -> SCST trên từng modality ----------
import random, gc, json
from configs.config import CFG as cfg
from models.slt_transformer import SLTTransformer
from training.train_xe import train_xe, evaluate
from training.train_scst import train_scst

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

cfg.model.encoder_type = "transformer"
cfg.train.xe_epochs = XE_EPOCHS
cfg.train.rl_epochs = RL_EPOCHS
cfg.train.xe_batch_size = BATCH
cfg.train.xe_warmup_steps = 500          # subset nhỏ -> warmup ngắn để LR kịp lên
cfg.data.max_frames = MAX_FRAMES
cfg.data.vocab_size = VOCAB
cfg.device = "cuda" if torch.cuda.is_available() else "cpu"
WORK = "/kaggle/working"

# Tokenizer build 1 LẦN từ text train (dùng chung mọi modality -> so sánh CÔNG BẰNG cùng vocab)
tok = build_tokenizer_from_tsv(TSV["train"], TEXT_COL, WORK, VOCAB)
print("device:", cfg.device, "| vocab_size:", tok.vocab_size)

def eval_ckpt(ckpt, loader):
    if not ckpt or not os.path.exists(ckpt): return None
    m = SLTTransformer(cfg, tok.vocab_size, pose_dim=cfg.data.pose_dim, encoder_type="transformer")
    m.load_state_dict(torch.load(ckpt, map_location=cfg.device)["model"]); m = m.to(cfg.device)
    bleu, _, _ = evaluate(m, loader, tok, cfg)
    del m; torch.cuda.empty_cache()
    return round(float(bleu), 2)

RESULTS = {}
for mod in MODALITIES:
    print("\n" + "#" * 72 + "\n# MODALITY = %s  (feat_dim=%d)\n" % (mod, FEAT_DIM[mod]) + "#" * 72)
    cfg.data.pose_dim = FEAT_DIM[mod]
    log_dir = os.path.join(WORK, "feat_%s_subset%d" % (mod, int(SUBSET * 100)))
    os.makedirs(log_dir, exist_ok=True)

    tr, dv, te = make_loaders_feat(mod, tok, SUBSET)
    print("Train %d | Val %d | Test %d" % (len(tr.dataset), len(dv.dataset), len(te.dataset)))

    # --- Phase 1: XE ---
    set_seed(cfg.seed)
    model = SLTTransformer(cfg, tok.vocab_size, pose_dim=FEAT_DIM[mod], encoder_type="transformer")
    print("Params: %.2fM" % (sum(p.numel() for p in model.parameters()) / 1e6))
    train_xe(model, tr, dv, tok, cfg, log_dir)
    xe_ckpt = os.path.join(log_dir, "best_xe.pt")
    res = {"feat_dim": FEAT_DIM[mod], "train_n": len(tr.dataset),
           "xe_test_bleu4": eval_ckpt(xe_ckpt, te)}
    del model; gc.collect(); torch.cuda.empty_cache()

    # --- Phase 2: SCST (fresh model, load XE bên trong train_scst) ---
    if RUN_SCST and os.path.exists(xe_ckpt):
        set_seed(cfg.seed)
        model = SLTTransformer(cfg, tok.vocab_size, pose_dim=FEAT_DIM[mod], encoder_type="transformer")
        train_scst(model, tr, dv, tok, cfg, log_dir, xe_ckpt)
        best, last = os.path.join(log_dir, "best_rl.pt"), os.path.join(log_dir, "last_rl.pt")
        rl_ckpt = best if os.path.exists(best) else (last if os.path.exists(last) else None)
        # best_rl.pt chỉ tồn tại nếu SCST VƯỢT XE; nếu không, eval last_rl.pt để vẫn báo cáo được
        res["scst_test_bleu4"] = eval_ckpt(rl_ckpt, te)
        res["scst_beat_xe"] = os.path.exists(best)
        del model; gc.collect(); torch.cuda.empty_cache()

    RESULTS[mod] = res
    print("[%s] ->" % mod, res)

json.dump(RESULTS, open(os.path.join(WORK, "feat_results.json"), "w"), indent=2)
print("\nĐã lưu /kaggle/working/feat_results.json")

In [ ]:
# ---------- BẢNG + BIỂU ĐỒ so sánh i3d vs mediapipe ----------
import os, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

WORK = "/kaggle/working"
pct = int(SUBSET * 100)

# 1) BẢNG kết quả (DataFrame -> Kaggle render HTML đẹp), có cột Δ(SCST−XE)
rows = []
for mod, r in RESULTS.items():
    xe, sc = r.get("xe_test_bleu4"), r.get("scst_test_bleu4")
    rows.append({
        "modality": mod, "feat_dim": r["feat_dim"], "train_n": r["train_n"],
        "XE_BLEU4": xe, "SCST_BLEU4": sc,
        "Δ(SCST−XE)": (round(sc - xe, 2) if (xe is not None and sc is not None) else None),
        "SCST>XE": ("có" if r.get("scst_beat_xe") else "không") if sc is not None else "-",
    })
df = pd.DataFrame(rows)
display(Markdown(f"### Kết quả test BLEU-4 — i3d vs mediapipe (subset {pct}%)"))
display(df)
df.to_csv(os.path.join(WORK, "feat_compare_table.csv"), index=False)

# 2) BAR CHART nhóm: mỗi modality có cột XE + SCST, có SỐ trên cột
mods = list(RESULTS.keys())
xe_v = [RESULTS[m].get("xe_test_bleu4") for m in mods]
sc_v = [RESULTS[m].get("scst_test_bleu4") for m in mods]
x = np.arange(len(mods)); w = 0.38
fig, ax = plt.subplots(figsize=(6, 4))
for i, (name, vals, col) in enumerate([("XE", xe_v, "#4C72B0"), ("SCST", sc_v, "#C44E52")]):
    b = ax.bar(x + (i - 0.5) * w, [v or 0 for v in vals], w, label=name, color=col)
    for bx, v in zip(b, vals):
        if v is not None:
            ax.annotate("%.2f" % v, (bx.get_x() + bx.get_width() / 2, v),
                        ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(mods)
ax.set_ylabel("Test BLEU-4"); ax.set_title("i3d vs mediapipe — subset %d%%" % pct)
ax.legend(); ax.grid(alpha=0.3, axis="y")
_top = max([v for v in (xe_v + sc_v) if v is not None] or [1])
ax.set_ylim(0, _top * 1.18)
plt.savefig(os.path.join(WORK, "fig_feat_compare.png"), bbox_inches="tight", dpi=140)
plt.show()

# 3) BLEU dev theo epoch mỗi modality — RL rẽ nhánh TỪ đúng epoch chọn best_xe (warm-start)
def _dev_ys(path):
    if not os.path.exists(path): return None
    h = json.load(open(path, encoding="utf-8"))
    ys = [e.get("dev_bleu4") for e in h if e.get("dev_bleu4") is not None]
    return ys or None

for mod in mods:
    ld = os.path.join(WORK, "feat_%s_subset%d" % (mod, pct))
    xe_ys = _dev_ys(os.path.join(ld, "xe_history.json"))
    rl_ys = _dev_ys(os.path.join(ld, "rl_history.json"))
    if not xe_ys and not rl_ys:
        continue
    warm = max(range(len(xe_ys)), key=lambda i: xe_ys[i]) if xe_ys else 0
    fig, ax = plt.subplots(figsize=(7, 4))
    if xe_ys:
        ax.plot(range(len(xe_ys)), xe_ys, "-o", ms=3, color="#4C72B0", label="XE (dev)")
        ax.axvline(warm, ls="--", color="gray", alpha=0.7)
        ax.scatter([warm], [xe_ys[warm]], s=60, color="black", zorder=5)
        ax.annotate("warm-start ep%d\nBLEU %.2f" % (warm, xe_ys[warm]), (warm, xe_ys[warm]),
                    xytext=(6, -28), textcoords="offset points", fontsize=8)
    if rl_ys:
        base = xe_ys[warm] if xe_ys else None
        xs = list(range(warm, warm + len(rl_ys) + 1)) if base is not None else list(range(len(rl_ys)))
        ys = ([base] + rl_ys) if base is not None else rl_ys
        ax.plot(xs, ys, "-o", ms=3, color="#C44E52", label="SCST (dev)")
    ax.set_title("[%s] BLEU dev theo epoch — RL rẽ nhánh từ warm-start (subset %d%%)" % (mod, pct))
    ax.set_xlabel("Epoch"); ax.set_ylabel("Dev BLEU-4"); ax.legend(loc="lower right"); ax.grid(alpha=0.3)
    plt.savefig(os.path.join(WORK, "fig_feat_epoch_%s.png" % mod), bbox_inches="tight", dpi=140)
    plt.show()